# 03 — Integrar dos conjuntos de datos

**Taller de célula única CIAD**

Adaptado de *Introduction to scRNA-seq integration* de Seurat:
<https://satijalab.org/seurat/articles/integration_introduction>

Todo hasta ahora usó una sola muestra. Los proyectos reales casi nunca son así. En cuanto
tienes dos — dos pacientes, dos condiciones, dos corridas en días distintos — te encuentras
con el problema del que trata este notebook.

**El problema.** Las células forman clusters según *de dónde vienen* en lugar de según *qué
son*. Las células B del batch 1 y las células B del batch 2 caen en dos
clusters separados. Entonces los resultados y las conclusiones están mal por construcción, o están sesgados
hacia artefactos técnicos. Encuentras el número equivocado de tipos celulares, y tu
prueba de expresión diferencial encuentra el batch, no la biología.

**Los datos.** Dos conjuntos de datos de PBMCs de 10x Genomics:

| nombre que usamos | conjunto de datos | células | química |
|---|---|---|---|
| **batch 1** | `pbmc3k` | ~2,700 | v1 (2016) |
| **batch 2** | `pbmc_1k_v3` | ~1,200 | v3 (2018) |

De aquí en adelante los llamamos **batch 1** y **batch 2**. La versión de la química es
la razón por la que difieren, pero los nombres se quedan simples.

Mismo tejido, mismos tipos celulares, distinta tecnología y distinto día. Ese es un
batch effect real, no uno simulado — y a diferencia de un conjunto de datos con dos condiciones, sabemos
con certeza que cualquier separación por batch es técnica, porque la biología
es la misma.

**Qué hacemos**

1. construir un objeto a partir de dos conjuntos de datos, y ver el batch effect
2. corregirlo de tres formas: **CCA de Seurat**, **Harmony** y **Canek**
3. comparar las tres, a ojo y con un número
4. nombrar los clusters nosotros mismos, a partir de los marcadores
5. preguntar qué nos dice y qué no nos dice la comparación

Unos 60 minutos.

## Preparación

Este notebook usa un **archivo de preparación distinto** al de los notebooks 01 y 02.

Instala Canek además de Seurat, lo que toma varios minutos. Arranca la
primera celda y lee la introducción de arriba mientras corre.

In [ ]:
# This cell installs the packages we need. It is a shortcut for the workshop,
# so that nobody spends the class waiting for an install.
source("https://raw.githubusercontent.com/MartinLoza/CIAD_workshop_sc/main/setup/setup_canek.R")

In [ ]:
# Change R language to English, in case your computer uses another language.
Sys.setenv(LANGUAGE = "en")

# This is the normal way to load a package in R. You will write lines like
# these at the top of every script you make.
library(Seurat)
library(ggplot2)
library(dplyr)
library(patchwork)
library(Canek)
library(harmony)

# Size of every figure in this notebook, in inches. Change these two numbers
# if a plot looks too small or too large.
options(repr.plot.width = 10, repr.plot.height = 7)

In [ ]:
cat("Canek  ", as.character(packageVersion("Canek")), "\n")
cat("harmony", as.character(packageVersion("harmony")), "\n")

## 1. Dos conjuntos de datos

Descarga los dos, lee los dos, y lleva la cuenta de cuál es cuál.

In [ ]:
# batch 1 — the pbmc3k data from notebooks 01 and 02
download.file("https://cf.10xgenomics.com/samples/cell/pbmc3k/pbmc3k_filtered_gene_bc_matrices.tar.gz",
              "pbmc3k.tar.gz", quiet = TRUE)
untar("pbmc3k.tar.gz")

# batch 2 — a later run, with newer chemistry
download.file("https://cf.10xgenomics.com/samples/cell-exp/3.0.0/pbmc_1k_v3/pbmc_1k_v3_filtered_feature_bc_matrix.tar.gz",
              "pbmc1k.tar.gz", quiet = TRUE)
untar("pbmc1k.tar.gz")

In [ ]:
counts_batch1 <- Read10X("filtered_gene_bc_matrices/hg19")
counts_batch2 <- Read10X("filtered_feature_bc_matrix")

cat("batch 1:", nrow(counts_batch1), "genes x", ncol(counts_batch1), "cells\n")
cat("batch 2:", nrow(counts_batch2), "genes x", ncol(counts_batch2), "cells\n")

### La etiqueta de batch

Esta es la variable más importante de este notebook. Todo lo que sigue — lo que
graficamos, lo que corregimos, lo que medimos — depende de ella.

In [ ]:
batch1 <- CreateSeuratObject(counts_batch1, project = "batch1",
                             min.cells = 3, min.features = 200)
batch2 <- CreateSeuratObject(counts_batch2, project = "batch2",
                             min.cells = 3, min.features = 200)

# the batch label. Every later step reads this column.
batch1$batch <- "batch1"
batch2$batch <- "batch2"

cat("batch 1:", ncol(batch1), "cells\n")
cat("batch 2:", ncol(batch2), "cells\n")

### Control de calidad, por batch

El control de calidad se hace **dentro de cada batch**, nunca entre batches. Las dos químicas capturan
cantidades distintas de RNA, así que un solo umbral de counts quitaría en silencio muchas
más células de un batch que del otro. Eso sería un batch effect nuevo,
creado por nosotros.

In [ ]:
batch1[["percent.mt"]] <- PercentageFeatureSet(batch1, pattern = "^MT-")
batch2[["percent.mt"]] <- PercentageFeatureSet(batch2, pattern = "^MT-")

VlnPlot(batch1, features = c("nFeature_RNA", "percent.mt"), ncol = 2) +
  plot_annotation(title = "Batch 1")


In [ ]:
VlnPlot(batch2, features = c("nFeature_RNA", "percent.mt"), ncol = 2) +
  plot_annotation(title = "Batch 2")

Mira la diferencia en `nFeature_RNA` antes de filtrar. El batch 2 detecta muchos
más genes por célula, porque su química es más nueva. Esa brecha es el batch effect. Lo podemos ver
antes de cualquier análisis.

In [ ]:
before <- c(ncol(batch1), ncol(batch2))

batch1 <- subset(batch1, subset = nFeature_RNA > 200 & nFeature_RNA < 2500 & percent.mt < 5)
batch2 <- subset(batch2, subset = nFeature_RNA > 500 & nFeature_RNA < 5000 & percent.mt < 15)

cat("batch 1:", before[1], "->", ncol(batch1), "cells\n")
cat("batch 2:", before[2], "->", ncol(batch2), "cells\n")

### ✏️ Ejercicio 1

Los umbrales de arriba son distintos para los dos batches — 200–2500 genes y 5%
mitocondrial para el batch 1, 500–5000 y 15% para el batch 2.

Justifica o rechaza esa decisión. Regresa a ver las dos gráficas de violín. ¿Un solo
umbral compartido habría sido más honesto, o menos?

No hay que programar nada. Escribe tu respuesta en la siguiente celda y prepárate para defenderla.

*Tu respuesta:*

### Un objeto, dos layers

`merge()` combina los objetos. En Seurat v5 los counts se quedan en **layers
separados**, uno por batch — no se juntan por defecto.

Esa estructura de layers es sobre la que después trabaja `IntegrateLayers()`.

Nota primero la intersección de genes. Los dos conjuntos de datos se mapearon a construcciones
de referencia distintas, así que sus listas de genes difieren. Si hacemos el merge sin intersectar, los genes
faltantes se llenan con ceros. Eso crea una diferencia entre los batches que
no es real.

In [ ]:
shared <- intersect(rownames(batch1), rownames(batch2))

cat("genes in batch 1:", nrow(batch1), "\n")
cat("genes in batch 2:", nrow(batch2), "\n")
cat("shared          :", length(shared), "\n")

pbmc <- merge(
  batch1[shared, ],
  y = batch2[shared, ],
  add.cell.ids = c("batch1", "batch2")
)

pbmc

In [ ]:
# two layers, one per batch
Layers(pbmc[["RNA"]])

table(pbmc$batch)

## 2. Ver el batch effect

Corre el pipeline estándar, exactamente como en el notebook 02, y mira el resultado
coloreado por batch.

Como los layers están separados, la normalización y la selección de genes variables se
hacen por layer y después se combinan — esto es Seurat v5 manejando la estructura de batches
por ti.

In [ ]:
pbmc <- NormalizeData(pbmc, verbose = FALSE)
pbmc <- FindVariableFeatures(pbmc, verbose = FALSE)
pbmc <- ScaleData(pbmc, verbose = FALSE)
pbmc <- RunPCA(pbmc, verbose = FALSE)

pbmc <- RunUMAP(pbmc, dims = 1:30, reduction = "pca",
                reduction.name = "umap.unintegrated", verbose = FALSE)

DimPlot(pbmc, reduction = "umap.unintegrated", group.by = "batch") +
  ggtitle("no integration")

Las células se separan por batch, no por tipo celular.

Para estar seguros de que eso es técnico y no biológico, revisa un marcador. `MS4A1`
marca células B, y los dos batches contienen células B — así que si las células B están en dos
lugares separados, la separación es técnica.

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 5)
FeaturePlot(pbmc, reduction = "umap.unintegrated",
            features = c("MS4A1", "CD3E", "CD14", "PPBP"), ncol = 4)

Cada marcador aparece en **dos** lugares, uno por batch. Mismo tipo celular, partido en
dos solo por la tecnología. Eso es lo que la integración tiene que arreglar.

### Qué hace el clustering con esto

In [ ]:
pbmc <- FindNeighbors(pbmc, dims = 1:30, reduction = "pca", verbose = FALSE)
pbmc <- FindClusters(pbmc, resolution = 0.5, verbose = FALSE)

# how pure is each cluster, in batch terms?
round(prop.table(table(pbmc$seurat_clusters, pbmc$batch), margin = 1), 2)

Lee esa tabla así: para cada cluster, qué fracción vino de cada batch.

Un cluster de 0.92 / 0.08 es un artefacto de batch — es la versión de un
tipo celular de un solo batch. Si le dieras estos clusters a una prueba de expresión diferencial
obtendrías una lista larga de genes significativos que en realidad son la química (artefactos técnicos).

## 3. El método propio de Seurat

`IntegrateLayers()` es la puerta de entrada única de Seurat v5 para la integración. Le pasas
un método, le dices desde qué reduction empezar, y nombras la reduction que va a crear.

El método incluido en Seurat es **CCA**, análisis de correlación canónica. Busca
direcciones de variación que sean *compartidas* entre los batches. La idea es que la variación
compartida es biología, y la variación que se encuentra en un solo batch es técnica. Después
encuentra "anchors" — pares de células de batches distintos que son vecinas mutuas en
ese espacio compartido — y los usa para calcular la corrección.

Este es el método que usa la viñeta de Seurat. Es el más completo de los tres
y también el más lento: espera un par de minutos.

In [ ]:
pbmc <- IntegrateLayers(
  object         = pbmc,
  method         = CCAIntegration,
  orig.reduction = "pca",
  new.reduction  = "integrated.cca",
  verbose        = FALSE
)

Reductions(pbmc)

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 5)
pbmc <- RunUMAP(pbmc, dims = 1:30, reduction = "integrated.cca",
                reduction.name = "umap.cca", verbose = FALSE)

DimPlot(pbmc, reduction = "umap.cca", group.by = "batch") +
  ggtitle("Seurat CCA")

## 4. Harmony

La misma función, otro método — ese es el punto de `IntegrateLayers()`.

Harmony trabaja **directamente sobre el embedding del PCA**. No busca anchors
entre células: toma las coordenadas del PCA, las agrupa de forma suave, y de manera iterativa
empuja las células de cada batch hacia los centros de cluster compartidos, repitiendo hasta que los
batches se traslapan.

Tiene menos pasos que CCA, y es mucho más rápido.

In [ ]:
pbmc <- IntegrateLayers(
  object         = pbmc,
  method         = HarmonyIntegration,
  orig.reduction = "pca",
  new.reduction  = "harmony",
  verbose        = FALSE
)

Reductions(pbmc)

In [ ]:
pbmc <- RunUMAP(pbmc, dims = 1:30, reduction = "harmony",
                reduction.name = "umap.harmony", verbose = FALSE)

DimPlot(pbmc, reduction = "umap.harmony", group.by = "batch") +
  ggtitle("Harmony")

## 5. Canek

Un método distinto, aplicado en el mismo punto del pipeline.

Canek identifica **vecinos mutuos más cercanos** entre batches — pares de células
que son la coincidencia más cercana una de la otra cruzando la frontera entre batches, y que se toman
como el mismo tipo celular. A partir de esos pares estima la corrección, usando una
mezcla de un modelo lineal y uno no lineal.

Igual que con Harmony, lo corremos sobre el **embedding del PCA**, así que los dos son directamente
comparables: misma entrada, mismo tipo de salida, solo cambia la corrección.

Dos diferencias prácticas en cómo se llama:

- Canek no es un método de `IntegrateLayers`. Se llama directamente con
  `RunCanek()`, y lee el batch de una **columna de metadata** en lugar de los
  layers. Por eso primero unimos los layers.
- `correctEmbeddings = TRUE` es lo que hace que corrija el PCA en lugar de los
  valores de expresión. Lee la reduction `pca` existente y escribe una nueva
  llamada `canek`, sin tocar `pca` — por eso después todavía podemos graficar la
  versión sin corregir.

`pcaDim = 30` corresponde a las 30 dimensiones que le dimos a Harmony. Sin eso Canek
usaría todos los componentes del PCA, y la comparación no sería pareja.

In [ ]:
pbmc[["RNA"]] <- JoinLayers(pbmc[["RNA"]])

Layers(pbmc[["RNA"]])

In [ ]:
pbmc <- RunCanek(pbmc,
                 batches           = "batch",
                 correctEmbeddings = TRUE,
                 pcaDim            = 30)

Reductions(pbmc)

Una reduction nueva, `canek`, junto a `pca` y `harmony`. No se sobrescribió nada
y no se creó ningún assay nuevo — la corrección vive por completo en el
embedding, exactamente igual que la de Harmony.

Así que el paso que falta es el mismo que corrimos para Harmony: un UMAP de las coordenadas
corregidas.

In [ ]:
pbmc <- RunUMAP(pbmc, dims = 1:30, reduction = "canek",
                reduction.name = "umap.canek", verbose = FALSE)

DimPlot(pbmc, reduction = "umap.canek", group.by = "batch") +
  ggtitle("Canek")

## 6. Comparar las cuatro

Lado a lado, coloreadas por batch. Lo que quieres ver: los dos colores mezclados
en todas partes, y que la forma general siga mostrando grupos distintos.

Una gráfica como esta muestra las dos formas en que la corrección puede fallar. Corregir de menos
deja los batches separados. Corregir de más junta todo en un solo
grupo: los batches quedan mezclados, pero los tipos celulares desaparecieron.

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 10)

panel <- function(reduction, title, legend = FALSE) {
  p <- DimPlot(pbmc, reduction = reduction, group.by = "batch") + ggtitle(title)
  if (legend) p else p + theme(legend.position = "none")
}

panel("umap.unintegrated", "none") +
  panel("umap.cca",     "Seurat CCA") +
  panel("umap.harmony", "Harmony") +
  panel("umap.canek",   "Canek", legend = TRUE)

### ¿Sobrevivió la biología?

Mezclar batches es fácil. Revolver los datos también lo lograría. La prueba real es
si los tipos celulares siguen separados después.

In [ ]:
options(repr.plot.width = 14, repr.plot.height = 8)

FeaturePlot(pbmc, reduction = "umap.cca",
            features = c("MS4A1", "CD3E", "CD14", "PPBP"), ncol = 4) +
  plot_annotation(title = "Seurat CCA — each marker should now be in ONE place")

In [ ]:
FeaturePlot(pbmc, reduction = "umap.harmony",
            features = c("MS4A1", "CD3E", "CD14", "PPBP"), ncol = 4) +
  plot_annotation(title = "Harmony")

In [ ]:
FeaturePlot(pbmc, reduction = "umap.canek",
            features = c("MS4A1", "CD3E", "CD14", "PPBP"), ncol = 4) +
  plot_annotation(title = "Canek")

### ✏️ Ejercicio 2

Escoge otro par de marcadores y revísalo en los dos UMAPs integrados. Sugerencias:
`GNLY` y `NKG7` para células NK, `FCER1A` para células dendríticas, `CD8A` para células
T CD8.

¿El marcador está en un solo lugar, o en dos?

In [ ]:
# FeaturePlot(pbmc, reduction = "umap.harmony", features = c(______), ncol = 2)

## 7. Ponerle un número

El ojo no es confiable, y un UMAP es solo una proyección. Una métrica simple:

> Para cada célula, mira sus 30 vecinos más cercanos en el espacio corregido. ¿Qué
> fracción viene del *otro* batch?

Si los batches están perfectamente mezclados, esa fracción se acerca a la proporción que el otro batch
representa en los datos. Si los batches se quedan separados, se acerca a cero.

Esta es la idea detrás de métricas publicadas como kBET y LISI, reducida a
algo que puedes leer en diez líneas.

In [ ]:
mixing <- function(obj, reduction, batch = "batch", k = 30, dims = 1:30) {
  emb <- Embeddings(obj, reduction)[, dims]
  b   <- as.character(obj[[batch]][, 1])

  nn <- FNN::get.knn(emb, k = k)$nn.index
  # fraction of each cell's neighbours belonging to a different batch
  mean(rowMeans(matrix(b[nn], nrow = nrow(nn)) != b))
}

# what perfect mixing would look like: the chance two random cells differ
p  <- prop.table(table(pbmc$batch))
ideal <- 1 - sum(p^2)

cat(sprintf("%-14s %s\n", "method", "cross-batch neighbours"))
cat(sprintf("%-14s %.3f\n", "none",    mixing(pbmc, "pca")))
cat(sprintf("%-14s %.3f\n", "Seurat CCA", mixing(pbmc, "integrated.cca")))
cat(sprintf("%-14s %.3f\n", "Harmony", mixing(pbmc, "harmony")))
cat(sprintf("%-14s %.3f\n", "Canek",   mixing(pbmc, "canek")))
cat(sprintf("\n%-14s %.3f  (perfectly mixed)\n", "ideal", ideal))

### Lee esto con cuidado

Más alto es más mezclado. **No** es simplemente mejor.

Un método que destruyera toda la estructura biológica sacaría un valor cercano al ideal
y sería inútil. El número solo te dice si los batches se mezclaron; las gráficas de marcadores
de arriba te dicen si los tipos celulares sobrevivieron. Necesitas las dos cosas. Ninguna sola
es evidencia.

### ✏️ Ejercicio 3

Mira los batches **por cluster** en lugar de globalmente.

Haz clustering sobre el embedding de Harmony, y después reporta qué clusters siguen viniendo casi
por completo de un solo batch. Un cluster que sigue sin mezclarse después de la integración es donde
el método falló — y muchas veces es un tipo celular que de verdad está presente en un solo
batch.

Llena el espacio en blanco:

In [ ]:
pbmc <- FindNeighbors(pbmc, reduction = "harmony", dims = 1:30, verbose = FALSE)
pbmc <- FindClusters(pbmc, resolution = 0.5, verbose = FALSE)

round(prop.table(table(pbmc$seurat_clusters, pbmc$batch), margin = 1), 2)

# Which clusters are still dominated by one batch?
# threshold <- ______
# names(which(apply(prop.table(table(pbmc$seurat_clusters, pbmc$batch), 1), 1, max) > threshold))

### Clusters, antes y después

La recompensa práctica. Compara esta tabla con la de la sección 2.

In [ ]:
options(repr.plot.width = 12, repr.plot.height = 5)

DimPlot(pbmc, reduction = "umap.harmony",
        group.by = c("batch", "seurat_clusters"))

## 8. Nombrar los clusters

En el notebook 02 la lista de etiquetas te la dimos nosotros. Aquí la escribes tú.

Los clusters no son los mismos que en el notebook 02. Vienen de un conjunto distinto
de células, de dos batches en lugar de uno, y de un embedding integrado. Así que
los números de cluster son distintos, y su orden es distinto.

Los genes marcadores son los mismos, porque este es el mismo tejido: PBMCs de un
donante sano, sin tratamiento. Solo cambió la química.

In [ ]:
# Markers are computed on the ORIGINAL counts, in the RNA assay,
# never on the corrected embedding. This is the rule from the previous section.
DefaultAssay(pbmc) <- "RNA"

# About a minute.
markers <- FindAllMarkers(pbmc, only.pos = TRUE, verbose = FALSE)

head(markers, 5)

Los clusters que tienes que etiquetar, y los genes que separan a cada uno:

In [ ]:
# write your labels in this order
levels(pbmc)

# the three strongest markers of each cluster
top3 <- markers %>%
  group_by(cluster) %>%
  slice_max(avg_log2FC, n = 3) %>%
  summarise(genes = paste(gene, collapse = ", "))

as.data.frame(top3)

### ✏️ Ejercicio 4

Nombra cada cluster. Compara los marcadores impresos arriba con esta tabla, la misma
que la del notebook 02:

| marcadores | tipo celular |
|---|---|
| `IL7R`, `CCR7` | T CD4 naive |
| `IL7R`, `S100A4` | T CD4 de memoria |
| `CD14`, `LYZ` | monocitos CD14+ |
| `MS4A1` | B |
| `CD8A` | T CD8 |
| `FCGR3A`, `MS4A7` | monocitos FCGR3A+ |
| `GNLY`, `NKG7` | NK |
| `FCER1A`, `CST3` | células dendríticas |
| `PPBP` | plaquetas |

Los nueve marcadores sobreviven a la intersección de genes, así que todos están
disponibles aquí.

Dos cosas que hay que cuidar:

- Escribe una etiqueta por cluster, en el orden en que `levels(pbmc)` los imprimió.
- Dos clusters pueden recibir la misma etiqueta. Eso está permitido. Normalmente significa que la
  resolución partió un tipo celular en dos.

Llena el espacio en blanco, y después quita el `#` de las tres líneas de abajo:

In [ ]:
# new_ids <- c(______)

# names(new_ids) <- levels(pbmc)
# pbmc <- RenameIdents(pbmc, new_ids)
# pbmc$cell_type <- Idents(pbmc)

Ahora la recompensa de todo el notebook. La misma gráfica, separada por batch.

Si la integración funcionó y tus etiquetas están bien, cada tipo celular aparece en
**los dos** paneles. Antes de la integración cada tipo celular estaba en dos lugares separados
de un mismo panel.

In [ ]:
options(repr.plot.width = 14, repr.plot.height = 6)

# This runs whether or not you named the clusters. Without names it shows the
# cluster numbers instead.
DimPlot(pbmc, reduction = "umap.harmony", label = TRUE, repel = TRUE,
        split.by = "batch") + NoLegend()

## 9. ¿Qué método deberías usar?

Para un batch effect simple entre corridas del mismo tejido, la mayoría de los métodos actuales
funcionan, y las diferencias que ves aquí son más pequeñas
que las diferencias causadas por tus umbrales de control de calidad.

Lo que sí importa:

- **Integra para visualizar y para hacer clustering. Haz la expresión diferencial sobre los
  datos originales**, con el batch como covariable. A los valores corregidos se les quitó
  variación por diseño, así que los p-values calculados sobre ellos no son confiables.
  Por eso mantuvimos intacto el assay `RNA`.
- **Revisa que la biología haya sobrevivido.** Siempre. Un UMAP bien mezclado no prueba nada
  por sí solo.
- **Pregúntate si deberías integrar siquiera.** Si un tipo celular de verdad está
  presente en un batch y ausente en el otro, la integración va a tratar de mezclarlo
  de todos modos — y puede inventar una correspondencia que no existe.

### ✏️ Ejercicio 5

Integramos dos batches del *mismo* tejido, así que sabíamos que cualquier separación era
técnica.

Supón que en vez de eso los batches fueran *control* y *tratado*, y que el tratamiento
cambiara qué tipos celulares están presentes. ¿Qué le haría la integración a esa
diferencia, y cómo la distinguirías de un batch effect?

No hay que programar nada. Esta es la pregunta que hay que pensar antes de integrar tus propios
datos.

*Tu respuesta:*

## Qué hicimos

- Construimos un objeto a partir de dos conjuntos de datos reales y vimos el batch effect en los datos
  crudos, en el UMAP, en los marcadores y en la tabla de composición de los clusters.
- Lo corregimos de tres formas, todas actuando sobre el embedding del PCA: CCA de Seurat y
  Harmony a través de `IntegrateLayers()`, Canek a través de
  `RunCanek(correctEmbeddings = TRUE)`.
- Los comparamos a ojo y con un puntaje de mezcla de vecinos, y dijimos de forma explícita por qué
  ese puntaje no es suficiente por sí solo.

---

### Respuestas

<details>
<summary>Clic para desplegar</summary>

**Ejercicio 1**

Umbrales distintos son la decisión correcta, y las gráficas de violín son la
justificación: el batch 2 detecta más o menos el doble de genes por célula. Un límite
superior compartido de 2,500 genes quitaría una fracción grande de células sanas del batch 2
y casi ninguna del batch 1 — convirtiendo un paso de control de calidad en un batch effect. El control de calidad pregunta
"¿esto es una célula real?", y lo que cuenta como real depende de la química.

Una mejor versión de "un solo umbral compartido" es un cuantil: quedarse con el 95% central
*dentro de cada batch*.

**Ejercicio 2**

```r
FeaturePlot(pbmc, reduction = "umap.harmony", features = c("GNLY", "NKG7"), ncol = 2)
```

**Ejercicio 3**

```r
threshold <- 0.9
comp <- prop.table(table(pbmc$seurat_clusters, pbmc$batch), 1)
names(which(apply(comp, 1, max) > threshold))
```

Cualquier cluster que siga arriba de 0.9 después de la integración vale la pena mirarlo directamente.
A veces el método falló; a veces el tipo celular de verdad está en un solo batch,
y en ese caso dejarlo sin mezclar es el comportamiento correcto.

**Ejercicio 4**

No hay un vector correcto único, porque tus números de cluster dependen de la
resolución y del número de dimensiones. Lo que importa es el método:

```r
levels(pbmc)   # el orden en el que hay que escribir las etiquetas

new_ids <- c("Naive CD4 T", "CD14+ Mono", "Memory CD4 T", "B", "CD8 T",
             "FCGR3A+ Mono", "NK", "DC", "Platelet")   # el tuyo será distinto

names(new_ids) <- levels(pbmc)
pbmc <- RenameIdents(pbmc, new_ids)
pbmc$cell_type <- Idents(pbmc)
```

Si `RenameIdents()` se queja, tu vector no tiene la misma longitud que
`levels(pbmc)`. Cuenta los clusters otra vez.

Un cluster que no puedes nombrar vale la pena reportarlo, no esconderlo. Muchas veces es un
cluster de doublets, o un tipo celular presente en un solo batch.

**Ejercicio 5**

La integración no puede distinguir "este batch difiere técnicamente" de "este batch
difiere biológicamente" — solo ve que los batches difieren, y su trabajo es
quitar esa diferencia. Si el tratamiento cambió la composición de tipos celulares, la integración
va a empujar esas células juntas y puede esconder el efecto que estás estudiando.

Formas de distinguirlos:

- Los tipos celulares compartidos por las dos condiciones deberían alinearse; una población
  genuinamente específica de una condición no debería. Si *todo* se alinea perfectamente,
  sospecha que hubo sobrecorrección.
- Guarda los datos sin corregir y revisa si la diferencia se ve ahí.
- Haz la estadística sobre los counts sin corregir con la condición como covariable.
  La integración es para ver, no para probar.

</details>